# Sistem Prediksi DBD - Google Colab

Random Forest untuk prediksi Demam Berdarah Dengue (DBD)

**Cara pakai:** Jalankan semua cell dari atas ke bawah.
Setelah cell terakhir, klik link URL yang muncul untuk membuka aplikasi.

## 1. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/Suwardi87/prediksi-DBD.git /content/prediksi-DBD
%cd /content/prediksi-DBD/python_app
!pip install -q -r requirements.txt pyngrok

## 2. Setup Database (SQLite) & Import Data dari Excel

In [ ]:
import os, sys
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

from app import create_app, db
from app.models import User, PasienDBD, KasusBulanan
from werkzeug.security import generate_password_hash
import pandas as pd
from datetime import date
from collections import defaultdict
from app.ml_model import get_risk_level

app = create_app()

with app.app_context():
    db.create_all()

    # Seed users
    for uname, pw, role, nama in [
        ('admin', 'admin123', 'admin', 'Administrator'),
        ('petugas', 'petugas123', 'petugas', 'Petugas Kesehatan')]:
        if not User.query.filter_by(username=uname).first():
            db.session.add(User(
                username=uname, password=generate_password_hash(pw),
                nama_lengkap=nama, role=role, status='aktif'))
    db.session.commit()

    # Import 163 pasien dari Excel
    EXCEL = '/content/prediksi-DBD/Data DBD 15 Sampel.xlsx'
    df = pd.read_excel(EXCEL, sheet_name='Data_DBD')

    BULAN_KASUS = {
        'Januari': 12, 'Februari': 7, 'Maret': 14, 'April': 4,
        'Mei': 13, 'Juni': 15, 'Juli': 10, 'Agustus': 13,
        'September': 18, 'Oktober': 33, 'November': 21, 'Desember': 3
    }
    BULAN_ORDER = list(BULAN_KASUS.keys())
    reverse_map = defaultdict(list)
    for b, k in BULAN_KASUS.items():
        reverse_map[k].append(b)

    patients_by_kasus = defaultdict(list)
    for _, row in df.iterrows():
        patients_by_kasus[int(row['Jumlah Kasus Perbulan'])].append({
            'nama': str(row['Nama']).strip(),
            'usia': int(row['Usia']),
            'lama_rawat': int(row['Lama Rawat Inap']),
            'jk': 'L' if int(row['Jenis Kelamin']) == 1 else 'P',
        })

    tahun = 2024
    day_counter = {}
    p_no = 0
    for kasus_val in sorted(patients_by_kasus.keys()):
        patients = patients_by_kasus[kasus_val]
        bulans = reverse_map[kasus_val]
        mid = len(patients) // 2 if len(bulans) > 1 else len(patients)
        for i, p in enumerate(patients):
            p_no += 1
            bulan = bulans[0] if i < mid or len(bulans) == 1 else bulans[1]
            month_num = BULAN_ORDER.index(bulan) + 1
            day_counter[bulan] = day_counter.get(bulan, 0) + 2
            day = min(day_counter[bulan], 28)
            db.session.add(PasienDBD(
                no_rm=f'RM-{tahun}-{p_no:04d}', nama_pasien=p['nama'],
                usia=p['usia'], jenis_kelamin=p['jk'], alamat='Kab. Agam',
                tanggal_masuk=date(tahun, month_num, day),
                lama_rawat=p['lama_rawat'], bulan=bulan, tahun=tahun))
    db.session.commit()

    # Kasus bulanan
    for bulan, jumlah in BULAN_KASUS.items():
        db.session.add(KasusBulanan(
            bulan=bulan, tahun=tahun, jumlah_kasus=jumlah,
            jumlah_sembuh=0, jumlah_meninggal=0,
            tingkat_risiko=get_risk_level(jumlah)))
    db.session.commit()

    print(f'Pasien: {PasienDBD.query.count()}')
    print(f'Kasus Bulanan: {KasusBulanan.query.count()}')
    print(f'Login: admin/admin123 atau petugas/petugas123')

## 3. Jalankan Flask + Ngrok (Public URL)

In [ ]:
import os, sys
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

import threading, time
from pyngrok import ngrok
from app import create_app

app = create_app()

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(3)

public_url = ngrok.connect(5000)
line = '=' * 60
print(line)
print('  APLIKASI BERHASIL DIJALANKAN!')
print(line)
print(f'  Buka link ini di browser: {public_url}')
print('  Login: admin / admin123')
print('         petugas / petugas123')
print(line)

## 4. Training Model (Opsional)

Jalankan cell ini setelah login ke aplikasi untuk melatih model Random Forest.

In [ ]:
import os, sys, json
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

from app import create_app, db
from app.models import PasienDBD, KasusBulanan, ModelEvaluasi
from app.ml_model import prepare_training_data, train_model
from datetime import datetime

app = create_app()
with app.app_context():
    pasien_list = PasienDBD.query.all()
    kasus_dict = {}
    for kb in KasusBulanan.query.all():
        kasus_dict[(kb.bulan, kb.tahun)] = kb.jumlah_kasus

    df = prepare_training_data(pasien_list, kasus_dict)
    result = train_model(df, n_estimators=5, random_state=42)

    eval_record = ModelEvaluasi(
        tanggal_training=datetime.now(),
        accuracy=result['metrics']['accuracy'],
        precision_score=result['metrics']['precision_weighted'],
        recall_score=result['metrics']['recall_weighted'],
        f1_score=result['metrics']['f1_score_weighted'],
        mae=result['metrics']['mae'],
        rmse=result['metrics']['rmse'],
        r2_score=result['metrics']['r2_score'],
        n_estimators=5,
        confusion_matrix=json.dumps(result['confusion_matrix']),
        feature_importance=json.dumps(result['feature_importance']),
        model_path='models/random_forest_model.pkl')
    db.session.add(eval_record)
    db.session.commit()

    m = result['metrics']
    print(f'Accuracy:  {m["accuracy"]:.2%}')
    print(f'Precision: {m["precision_weighted"]:.2%}')
    print(f'Recall:    {m["recall_weighted"]:.2%}')
    print(f'F1 Score:  {m["f1_score_weighted"]:.2%}')
    print(f'MAE:       {m["mae"]:.4f}')
    print(f'RMSE:      {m["rmse"]:.4f}')
    print(f'R2 Score:  {m["r2_score"]:.4f}')
    print('Model berhasil disimpan! Buka aplikasi untuk melihat hasilnya.')